# Validate Train Corpus

In [1]:
from pathlib import Path

In [60]:
JA_PATH = Path('train/ja_train')
JA_DEV = Path('dev/ja_dev')

## Annotation Format

### Train

In [13]:
for f in JA_PATH.iterdir():
    if f.suffix != '.ann':
        continue
    with f.open() as anno:
        for line in anno:
            if line.startswith("T"):
                fields = line.strip().split("\t")
                if len(fields) != 3 or not fields[1].strip() or not fields[2].strip():
                    print(f.name, line.strip())
                else:
                    subfields = fields[1].split(" ")
                    if len(subfields) !=3 or subfields[0] not in ['DRUG', 'DISORDER', 'FUNCTION']:
                        print(f.name, line.strip())
                    else:
                        try:
                            start = int(subfields[1])
                            end = int(subfields[2])
                        except ValueError:
                            print(f.name, line.strip())
                    

ja_twjp_200-220_1.ann T15	DISORDER 47 49;52 55	体重 減った


### Dev

In [61]:
for f in JA_DEV.iterdir():
    if f.suffix != '.ann':
        continue
    with f.open() as anno:
        for line in anno:
            if line.startswith("T"):
                fields = line.strip().split("\t")
                if len(fields) != 3 or not fields[1].strip() or not fields[2].strip():
                    print(f.name, line.strip())
                else:
                    subfields = fields[1].split(" ")
                    if len(subfields) !=3 or subfields[0] not in ['DRUG', 'DISORDER', 'FUNCTION']:
                        print(f.name, line.strip())
                    else:
                        try:
                            start = int(subfields[1])
                            end = int(subfields[2])
                        except ValueError:
                            print(f.name, line.strip())
                    

ja_twjp_060-080_2.ann T4	DISORDER 67 69;72 74	食欲 減退
ja_twjp_320-340_15.ann T9	DISORDER 70 72;86 90	食欲 マイナス


## Validate NER

In [3]:
corpus = {}
for f in JA_PATH.iterdir():
    if f.suffix == '.txt':
        with f.open() as text:
            fields = text.readlines()[0].split(':')
            keyid = fields[0]
            offset = len(keyid)+1
            data = ':'.join(fields[1:])
            if f.stem in corpus:
                corpus[f.stem]['text'] = data
                corpus[f.stem]['offset'] = len(keyid) + 1
            else:
                corpus[f.stem] = {'text': data,
                                  'offset': len(keyid) + 1,
                                  'ann': []}
    elif f.suffix == '.ann':
        with f.open() as anno:
            for line in anno:
                if line.startswith("T"):
                    fields = line.strip().split("\t")
                    if f.stem in corpus:
                        corpus[f.stem]['ann'].append((fields[1], fields[2]))
                    else:
                        corpus[f.stem] = {'ann': [(fields[1], fields[2])]}

In [62]:
corpus_dev = {}
for f in JA_DEV.iterdir():
    if f.suffix == '.txt':
        with f.open() as text:
            fields = text.readlines()[0].split(':')
            keyid = fields[0]
            offset = len(keyid)+1
            data = ':'.join(fields[1:])
            if f.stem in corpus_dev:
                corpus_dev[f.stem]['text'] = data
                corpus_dev[f.stem]['offset'] = len(keyid) + 1
            else:
                corpus_dev[f.stem] = {'text': data,
                                  'offset': len(keyid) + 1,
                                  'ann': []}
    elif f.suffix == '.ann':
        with f.open() as anno:
            for line in anno:
                if line.startswith("T"):
                    fields = line.strip().split("\t")
                    if f.stem in corpus_dev:
                        corpus_dev[f.stem]['ann'].append((fields[1], fields[2]))
                    else:
                        corpus_dev[f.stem] = {'ann': [(fields[1], fields[2])]}

### Span

#### Train

In [16]:
for file_name, annotation in corpus.items():
    for info, ner in annotation['ann']:
        fields = info.split(' ')
        if len(fields) != 3:
            continue
        start = int(fields[1]) - annotation['offset']
        end = int(fields[2]) - annotation['offset']
        if annotation['text'][start: end] != ner:
            print(file_name, info, ner)

#### Dev

In [63]:
for file_name, annotation in corpus_dev.items():
    for info, ner in annotation['ann']:
        fields = info.split(' ')
        if len(fields) != 3:
            continue
        start = int(fields[1]) - annotation['offset']
        end = int(fields[2]) - annotation['offset']
        if annotation['text'][start: end] != ner:
            print(file_name, info, ner)

### Content

In [41]:
import unicodedata

In [58]:
def is_unwanted(c):
    # https://stackoverflow.com/a/60983873/11451863
    return (unicodedata.category(c).startswith("C") or  # others
            unicodedata.category(c).startswith("P") or  # punctuations
            unicodedata.category(c).startswith("S") or  # symbols
            unicodedata.category(c).startswith("Z")  # separators 
           )

#### Train

In [59]:
for file_name, annotation in corpus.items():
    for info, ner in annotation['ann']:
        for c in set(ner):
            if is_unwanted(c):
                print(c, file_name, info, ner)
                break

㎎ ja_twjp_160-180_14 DRUG 121 130 レクサプロ錠10㎎
. ja_twjp_400-420_4 DRUG 44 54 エチゾラム0.5mg
〜 ja_twjp_400-420_11 DISORDER 89 96 軽〜くめまい感
… ja_twjp_240-260_8 DRUG 140 143 エチ…
( ja_twjp_460-480_18 DISORDER 20 27 パニック障害(
… ja_twjp_180-200_7 DRUG 131 136 アモキサ…
  ja_twjp_200-220_1 DISORDER 47 49;52 55 体重 減った
💦 ja_twjp_340-360_14 FUNCTION 144 152 怖くなってきた💦
) ja_twjp_320-340_4 DRUG 109 115 レクサプロ)
） ja_twjp_060-080_4 DRUG 44 66 SSRI（選択的セロトニン再取り込み阻害薬）
… ja_twjp_020-040_0 DRUG 36 43 れくさ…ぷろ…
😢 ja_twjp_440-460_19 DISORDER 73 78 辛かった😢
… ja_twjp_280-300_1 DRUG 140 143 エチ…


#### Dev

In [64]:
for file_name, annotation in corpus_dev.items():
    for info, ner in annotation['ann']:
        for c in set(ner):
            if is_unwanted(c):
                print(c, file_name, info, ner)
                break

  ja_twjp_060-080_2 DISORDER 67 69;72 74 食欲 減退
) ja_twjp_340-360_0 DRUG 51 56 SSRI)
. ja_twjp_000-020_13 DRUG 40 55 抗不安薬ｿﾗﾅｯｸｽ0.4mg
  ja_twjp_120-140_11 DISORDER 50 63 普段 触れる物まで触れへん
  ja_twjp_120-140_11 DISORDER 89 105 長い時間かけて 手を洗わなあかん
  ja_twjp_320-340_15 DISORDER 70 72;86 90 食欲 マイナス
